In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install chembl_webresource_client pandas rdkit scikit-learn openpyxl

In [ ]:
import pandas as pd
import numpy as np
from chembl_webresource_client.new_client import new_client
import warnings
warnings.filterwarnings("ignore")
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split

In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

activity = new_client.activity

query = activity.filter(
    target_chembl_id="CHEMBL203",
    standard_type="IC50",
    standard_relation="="
).only([
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_value",
    "standard_units"
])

records = []

for i, rec in enumerate(query):

    records.append(rec)

    if i % 1000 == 0:
        print(f"Downloaded {i} records")


data = pd.DataFrame(records)

print(data.shape)

In [ ]:
# Keep only existing columns

data = data[
    [
        "molecule_chembl_id",
        "canonical_smiles",
        "standard_value",
        "standard_units"
    ]
]

# Remove missing values
data = data.dropna(
    subset=["canonical_smiles", "standard_value"]
)

# Convert activity to numeric
data["standard_value"] = pd.to_numeric(
    data["standard_value"],
    errors="coerce"
)

data = data.dropna(subset=["standard_value"])

print("After basic cleaning:", data.shape)

In [ ]:
print(data.columns)

In [ ]:

normalizer = rdMolStandardize.Normalizer()
uncharger = rdMolStandardize.Uncharger()
largest_fragment_chooser = rdMolStandardize.LargestFragmentChooser()

allowed_atoms = {
    "C", "H", "N", "O", "S", "P",
    "F", "Cl", "Br", "I"
}


def standardize_smiles(smiles):

    try:
        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            return None

        # Remove salts
        mol = largest_fragment_chooser.choose(mol)

        # Normalize
        mol = normalizer.normalize(mol)

        # Neutralize
        mol = uncharger.uncharge(mol)

        # Canonical smiles
        smiles = Chem.MolToSmiles(mol, canonical=True)

        mol = Chem.MolFromSmiles(smiles)

        # Molecular weight filter
        mw = Descriptors.MolWt(mol)

        if mw < 100 or mw > 1000:
            return None

        # Allowed atom filter
        atoms = set(atom.GetSymbol() for atom in mol.GetAtoms())

        if not atoms.issubset(allowed_atoms):
            return None

        return smiles

    except:
        return None


data["clean_smiles"] = data["canonical_smiles"].apply(
    standardize_smiles
)

data = data.dropna(subset=["clean_smiles"])

print("After SMILES standardization:", data.shape)



In [ ]:
def convert_to_nm(value, unit):

    if unit == "nM":
        return value

    elif unit == "uM":
        return value * 1000

    elif unit == "mM":
        return value * 1_000_000

    elif unit == "pM":
        return value / 1000

    else:
        return np.nan


data["IC50_nM"] = data.apply(
    lambda x: convert_to_nm(
        x["standard_value"],
        x["standard_units"]
    ),
    axis=1
)

data = data.dropna(subset=["IC50_nM"])

# Remove zero or negative values
data = data[data["IC50_nM"] > 0]


# ============================================
# STEP 5: CREATE pIC50
# ============================================

# Convert nM to M
data["IC50_M"] = data["IC50_nM"] * 1e-9

# pIC50 = -log10(IC50 in molar)
data["pIC50"] = -np.log10(data["IC50_M"])


# ============================================
# STEP 6: CREATE CLASS LABELS
# Active if pIC50 >= 6
# ============================================

data["activity_class"] = data["pIC50"].apply(
    lambda x: 1 if x >= 6 else 0
)

In [ ]:
data = (
    data.groupby(
        ["molecule_chembl_id", "clean_smiles"],
        as_index=False
    )
    .agg({
        "pIC50": "mean",
        "activity_class": "max"
    })
)

print("After duplicate removal:", data.shape)


# ============================================
# STEP 8: GENERATE MURCKO SCAFFOLDS
# ============================================

def get_scaffold(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    scaffold = MurckoScaffold.MurckoScaffoldSmiles(
        mol=mol
    )

    return scaffold


data["scaffold"] = data["clean_smiles"].apply(
    get_scaffold
)

data = data.dropna(subset=["scaffold"])

In [ ]:
cleaned_file = "EGFR_CHEMBL203_cleaned_data.xlsx"

data.to_excel(cleaned_file, index=False)

print(f"Cleaned dataset saved as: {cleaned_file}")


In [ ]:
scaffolds = data["scaffold"].unique()

# Split scaffolds
train_scaffolds, temp_scaffolds = train_test_split(
    scaffolds,
    test_size=0.30,
    random_state=42
)

valid_scaffolds, test_scaffolds = train_test_split(
    temp_scaffolds,
    test_size=0.67,
    random_state=42
)

# Create splits
train_df = data[
    data["scaffold"].isin(train_scaffolds)
]

valid_df = data[
    data["scaffold"].isin(valid_scaffolds)
]

test_df = data[
    data["scaffold"].isin(test_scaffolds)
]

print("\nDATA SPLITS")
print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)


# ============================================
# STEP 11: SAVE SPLITS
# ============================================

with pd.ExcelWriter(
    "EGFR_CHEMBL203_split_data.xlsx"
) as writer:

    train_df.to_excel(
        writer,
        sheet_name="Train",
        index=False
    )

    valid_df.to_excel(
        writer,
        sheet_name="Validation",
        index=False
    )

    test_df.to_excel(
        writer,
        sheet_name="Test",
        index=False
    )

print("\nSplit datasets saved to:")
print("EGFR_CHEMBL203_split_data.xlsx")

In [ ]:
# ============================================
# SAVE FILES TO KAGGLE WORKING DIRECTORY
# ============================================

kaggle_path = "/kaggle/working/"


# Save cleaned dataset
cleaned_file = kaggle_path + "EGFR_CHEMBL203_cleaned_data.xlsx"

data.to_excel(cleaned_file, index=False)


# Save split datasets
split_file = kaggle_path + "EGFR_CHEMBL203_split_data.xlsx"

with pd.ExcelWriter(split_file) as writer:

    train_df.to_excel(
        writer,
        sheet_name="Train",
        index=False
    )

    valid_df.to_excel(
        writer,
        sheet_name="Validation",
        index=False
    )

    test_df.to_excel(
        writer,
        sheet_name="Test",
        index=False
    )


# ============================================
# CLASS DISTRIBUTION
# ============================================

print("\nCLASS DISTRIBUTION")

print("\nTrain")
print(train_df["activity_class"].value_counts())

print("\nValidation")
print(valid_df["activity_class"].value_counts())

print("\nTest")
print(test_df["activity_class"].value_counts())


# ============================================
# FINAL OUTPUT FILES
# ============================================

print("\nFILES CREATED IN KAGGLE WORKING DIRECTORY:")

print(cleaned_file)
print(split_file)


# ============================================
# OPTIONAL CHECK
# ============================================

import os

print("\nFILES AVAILABLE:")

print(os.listdir("/kaggle/working/"))

In [ ]:
import pandas as pd

df = pd.read_excel("/kaggle/input/datasets/sherongeorge/egfr-dataset/EGFR_CHEMBL203_cleaned_data.xlsx")
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

train_df, external_df = train_test_split(df, test_size=0.3, random_state=42)

print("Train:", train_df.shape)
print("External (your dataset):", external_df.shape)

In [ ]:
external_df.to_csv("bindingdb_clean.csv", index=False)

In [ ]:
external_df = external_df.rename(columns={
    'clean_smiles': 'SMILES'
})

In [ ]:
from rdkit import Chem
import pickle

def mol_to_graph_simple(smiles, label):
    mol = Chem.MolFromSmiles(smiles)

    nodes = []
    for atom in mol.GetAtoms():
        nodes.append([
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetFormalCharge(),
            int(atom.GetIsAromatic())
        ])

    edges = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edges.append((i, j))
        edges.append((j, i))

    return {
        "nodes": nodes,
        "edges": edges,
        "label": label
    }
external_df = external_df.rename(columns={
    'clean_smiles': 'SMILES'
})

graph_data = []

for _, row in external_df.iterrows():
    g = mol_to_graph_simple(row['SMILES'], row['pIC50'])
    graph_data.append(g)

# Save
with open("bindingdb_graph.pkl", "wb") as f:
    pickle.dump(graph_data, f)

print("Graph dataset saved successfully")